In [1]:
# Start from scratch!
# Don't use my package to do this!
from robustraster import dask_cluster_manager

#json_key = r"R:\SCRATCH\adrianom\credentials\earthengineapi\robust-raster-cecdcc4b5fba.json"
dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="custom", threads_per_worker=1, n_workers=8, memory_limit="2GB")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [2]:
import ee

ee.Authenticate()


Successfully saved authorization token.


In [ ]:
# Start from scratch!
# Don't use my package to do this!
from robustraster import dask_cluster_manager

json_key = r"R:\SCRATCH\adrianom\credentials\earthengineapi\robust-raster-cecdcc4b5fba.json"
dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="custom", threads_per_worker=1, n_workers=2, memory_limit="400MB")

In [2]:
# 2. Authenticate Google Earth Engine on all Dask workers.
from robustraster import dask_plugins

#ee_plugin = dask_plugins.EEPlugin(json_key)
ee_plugin = dask_plugins.EEPlugin()
dask_client = dask_cluster.get_dask_client
dask_client.register_plugin(ee_plugin)

{'tcp://127.0.0.1:53005': {'status': 'OK'},
 'tcp://127.0.0.1:53008': {'status': 'OK'},
 'tcp://127.0.0.1:53011': {'status': 'OK'},
 'tcp://127.0.0.1:53012': {'status': 'OK'},
 'tcp://127.0.0.1:53013': {'status': 'OK'},
 'tcp://127.0.0.1:53020': {'status': 'OK'},
 'tcp://127.0.0.1:53021': {'status': 'OK'},
 'tcp://127.0.0.1:53026': {'status': 'OK'}}

In [3]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.

from robustraster import dataset_manager
import ee
import json
import ee

"""
# Although we authenticated Google Earth Engine on our Dask workers, we also need to authenticate
# Google Earth Engine on our local machine!
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')
"""

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

California = ee.FeatureCollection("projects/robust-raster/assets/boundaries/California")
#WSDemo = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
Plumas = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
# This is a dictionary that my code requires. You don't have to touch anything here for demo purposes
# (although it should work with anything, in theory. Feel free to change it if you'd like).
# These parameters get stored and are used to generate the header information needed when we do the full
# run.

#vector_path = r"R:\Users\adrianom.UNR\Downloads\boundaries\Plumas_National_forest\Plumas_National_forest.shp"
#vector_path = r"R:\SCRATCH\adrianom\code\big-raster-helper\geometry\Export_Geometry.geojson"

# Parameters for Plumas
parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2018-05-08',
            'end_date': '2018-05-10',
            'vector_path': Plumas,
            'crs': 'EPSG:3310',
            'scale': 30,
            'map_function': prep_sr_l8
        }

earth_engine = dataset_manager.EarthEngineDataset(parameters)

FEATURECOLLECTION!


In [4]:
print(earth_engine.dataset)

<xarray.Dataset> Size: 264MB
Dimensions:  (time: 2, X: 4741, Y: 3481)
Coordinates:
  * time     (time) datetime64[ns] 16B 2018-05-09T18:38:10.899000 2018-05-09T...
  * X        (X) float64 38kB -1.459e+05 -1.458e+05 ... -3.69e+03 -3.66e+03
  * Y        (Y) float64 28kB 1.516e+05 1.516e+05 ... 2.559e+05 2.56e+05
Data variables:
    SR_B4    (time, X, Y) float32 132MB ...
    SR_B5    (time, X, Y) float32 132MB ...
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:  SR_B7,SR_B5,SR_B3
    visualization_2_max:    30000.0
    visualization_2_min:    0.0
    

In [ ]:
print(earth_engine.dataframe)

In [4]:
# 4. Design your function here! 
 
# My target audience are for users who want to work with
# data frames, so pandas data frames are the only data structures 
# that I support for writing functions. If you'd prefer working 
# with xarrays (or possible other data structures), submit an
# issue and let me know!

# For this demo, we do a basic NDVI calculation.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df.drop(columns=['SR_B4', 'SR_B5'])

In [5]:
#### USING MY EXPORT CLASS ####
# EXPORT LOCALLY #
from robustraster import udf_manager
from robustraster import export_manager

user_defined_func = udf_manager.UserFunctionHandler(user_function=compute_ndvi)
export_object = export_manager.ExportProcessor(user_defined_func, flag="GTiff", output_folder="test231", chunks={"time": 48, "X": 512, "Y": 256})

export_object.run_and_export_results(earth_engine)
export_object.export_vrt(earth_engine)

c:\anaconda3\envs\rredit\lib\site-packages\osgeo\gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


VRT file created successfully: test231\time_2018_05_09T18_38_10.899000000.vrt
VRT file created successfully: test231\time_2018_05_09T18_38_34.781000000.vrt


In [ ]:
#### USING MY EXPORT CLASS ####
# EXPORT TO GCS #
from robustraster import udf_manager
from robustraster import export_manager

user_defined_func = udf_manager.UserFunctionHandler(user_function=compute_ndvi)
gcs_json_key = r"R:\SCRATCH\adrianom\credentials\cloud_storage\robust-raster-0a2573087969.json"
export_object = export_manager.ExportProcessor(user_defined_func, flag="GCS", chunks={"time": 48, "X": 512, "Y": 512}, gcs_credentials=gcs_json_key, gcs_bucket='gears-bucket-88', gcs_folder="fun")

export_object.run_and_export_results(earth_engine)

Created bucket: gears-bucket-88
Created folder: fun
